# Systeme de Recommandation de Films - IA
## Hackathon Project | Python + SciPy + Matplotlib

Ce notebook simule des donnees utilisateurs et des films,
puis construit un moteur de recommandation complet avec analyses statistiques et visualisations.

---

## 1. Imports

In [11]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from scipy.stats import pearsonr, linregress, f_oneway, chisquare
from scipy.spatial.distance import cosine
from scipy.cluster.vq import kmeans, whiten, vq

print("Bibliotheques chargees.")

Bibliotheques chargees.


## 2. Generation des donnees

On genere automatiquement :
- une liste de films avec genre, annee et note moyenne
- des profils utilisateurs conformes au format exact du sujet :



In [12]:
random.seed(42)
np.random.seed(42)

# --- Liste de films disponibles ---
FILMS_SOURCE = [
    ("Inception",         "sci-fi",       2010),
    ("Interstellar",      "sci-fi",       2014),
    ("The Dark Knight",   "action",       2008),
    ("Mad Max Fury Road", "action",       2015),
    ("John Wick",         "action",       2014),
    ("The Matrix",        "sci-fi",       1999),
    ("Dune",              "sci-fi",       2021),
    ("Parasite",          "thriller",     2019),
    ("Gone Girl",         "thriller",     2014),
    ("Se7en",             "thriller",     1995),
    ("The Shawshank",     "drame",        1994),
    ("Forrest Gump",      "drame",        1994),
    ("The Godfather",     "drame",        1972),
    ("Whiplash",          "drame",        2014),
    ("The Hangover",      "comedie",      2009),
    ("Superbad",          "comedie",      2007),
    ("Get Out",           "horreur",      2017),
    ("Hereditary",        "horreur",      2018),
    ("Titanic",           "romance",      1997),
    ("La La Land",        "romance",      2016),
    ("Toy Story",         "animation",    1995),
    ("Spirited Away",     "animation",    2001),
    ("Free Solo",         "documentaire", 2018),
    ("13th",              "documentaire", 2016),
    ("Indiana Jones",     "aventure",     1981),
    ("Jurassic Park",     "aventure",     1993),
    ("Arrival",           "sci-fi",       2016),
    ("Prisoners",         "thriller",     2013),
    ("12 Angry Men",      "drame",        1957),
    ("Midsommar",         "horreur",      2019),
]

def generer_films(films_source):
    films = []
    for i, (titre, genre, annee) in enumerate(films_source):
        films.append({
            "id": i + 1,
            "titre": titre,
            "genre": genre,
            "annee": annee,
            "note_moyenne": round(random.uniform(3.8, 5.0), 1)
        })
    return films

NOMS = ["Aminata", "Kofi", "Fatou", "Ibrahim", "Mariam"]
PREFERENCES_PAR_USER = [
    ["sci-fi", "thriller"],
    ["action", "aventure"],
    ["romance", "drame"],
    ["comedie", "animation"],
    ["documentaire", "drame"],
]

def generer_utilisateurs(films, noms, preferences_list):
    utilisateurs = []
    date_debut = datetime(2024, 1, 1)

    for i, nom in enumerate(noms):
        prefs = preferences_list[i]
        films_dispo = [f for f in films if f["genre"] in prefs]
        nb_films = min(8, len(films_dispo))
        films_vus = random.sample(films_dispo, nb_films)

        watch_history = []
        for j, film in enumerate(films_vus):
            date_visionnage = date_debut + timedelta(days=random.randint(j*15, j*15+10))
            watch_history.append({
                "movie": film["titre"],
                "genre": film["genre"],
                "rating": random.randint(3, 5),
                "date": date_visionnage.strftime("%Y-%m-%d")
            })

        utilisateurs.append({
            "name": nom,
            "age": random.randint(20, 40),
            "preferences": prefs,
            "watch_history": watch_history
        })

    return utilisateurs

films = generer_films(FILMS_SOURCE)
utilisateurs = generer_utilisateurs(films, NOMS, PREFERENCES_PAR_USER)

print(f"{len(films)} films generes")
print(f"{len(utilisateurs)} utilisateurs generes")
print()

df_films = pd.DataFrame(films)
print("Apercu des films :")
print(df_films.head(6).to_string(index=False))
print()

df_users = pd.DataFrame([{
    "Name": u["name"],
    "Age": u["age"],
    "Preferences": ", ".join(u["preferences"]),
    "Films vus": len(u["watch_history"]),
    "Rating moyen": round(np.mean([h["rating"] for h in u["watch_history"]]), 2)
} for u in utilisateurs])
print("Apercu des utilisateurs :")
print(df_users.to_string(index=False))
print()

# Exemple conforme au format du sujet
print("Exemple de profil (format exact du sujet) :")
import json as _json
exemple = utilisateurs[0].copy()
exemple["watch_history"] = exemple["watch_history"][:1]
print(_json.dumps(exemple, indent=4, ensure_ascii=False))


30 films generes
5 utilisateurs generes

Apercu des films :
 id             titre  genre  annee  note_moyenne
  1         Inception sci-fi   2010           4.6
  2      Interstellar sci-fi   2014           3.8
  3   The Dark Knight action   2008           4.1
  4 Mad Max Fury Road action   2015           4.1
  5         John Wick action   2014           4.7
  6        The Matrix sci-fi   1999           4.6

Apercu des utilisateurs :
   Name  Age         Preferences  Films vus  Rating moyen
Aminata   34    sci-fi, thriller          8          3.88
   Kofi   27    action, aventure          5          4.40
  Fatou   32      romance, drame          7          3.71
Ibrahim   38  comedie, animation          4          4.75
 Mariam   28 documentaire, drame          7          4.57

Exemple de profil (format exact du sujet) :
{
    "name": "Aminata",
    "age": 34,
    "preferences": [
        "sci-fi",
        "thriller"
    ],
    "watch_history": [
        {
            "movie": "Inception"

## 3. Recommandation par genre

On filtre les films non encore vus qui correspondent aux genres preferes,
puis on les classe par note decroissante.


In [ ]:
def recommander_par_genre(utilisateur, films, top_n=3):
    prefs = utilisateur["preferences"]
    deja_vus = {h["movie"] for h in utilisateur["watch_history"]}

    candidats = [
        f for f in films
        if f["genre"] in prefs and f["titre"] not in deja_vus
    ]
    candidats.sort(key=lambda f: (prefs.index(f["genre"]), -f["note_moyenne"]))
    return candidats[:top_n]


print("Recommandations par genre")
for u in utilisateurs:
    recs = recommander_par_genre(u, films)
    print(f'{u["name"]} (preferences : {", ".join(u["preferences"])})')
    for r in recs:
        print(f'  - {r["titre"]:<25} [{r["genre"]:<13}] note : {r["note_moyenne"]}')
    print()


## 4. Recommandation narrative

On identifie le film le mieux note dans l'historique de l'utilisateur
et on suggere des films du meme genre avec un message explicatif.


In [ ]:
def recommandation_narrative(utilisateur, films, top_n=2):
    watch_history = utilisateur["watch_history"]
    meilleur = max(watch_history, key=lambda h: h["rating"])
    deja_vus = {h["movie"] for h in watch_history}

    candidats = [
        f for f in films
        if f["genre"] == meilleur["genre"] and f["titre"] not in deja_vus
    ]
    candidats.sort(key=lambda f: -f["note_moyenne"])

    messages = []
    for f in candidats[:top_n]:
        msg = (f"Puisque vous avez aime '{meilleur['movie']}' (rating {meilleur['rating']}/5), "
               f"vous pourriez apprecier '{f['titre']}' ({f['genre']}, {f['annee']}, {f['note_moyenne']}/5)")
        messages.append(msg)
    return messages


print("Recommandations narratives")
for u in utilisateurs:
    print(f'{u["name"]} :')
    for msg in recommandation_narrative(u, films):
        print(f"  -> {msg}")
    print()


## 5. Filtrage collaboratif (Pearson)

`scipy.stats.pearsonr` mesure la correlation entre les notes de deux utilisateurs
sur les films qu'ils ont vus en commun.
On trouve ensuite l'utilisateur le plus similaire et on recommande ses films bien notes.


In [ ]:
def similarite_pearson(u1, u2):
    h1 = {h["movie"]: h["rating"] for h in u1["watch_history"]}
    h2 = {h["movie"]: h["rating"] for h in u2["watch_history"]}
    communs = list(set(h1) & set(h2))
    if len(communs) < 2:
        return 0.0
    r, _ = pearsonr([h1[f] for f in communs], [h2[f] for f in communs])
    return round(r, 3)


def recommander_collaboratif(utilisateur, tous_utilisateurs, top_n=3):
    similarities = {
        u["name"]: similarite_pearson(utilisateur, u)
        for u in tous_utilisateurs if u["name"] != utilisateur["name"]
    }
    meilleur_nom = max(similarities, key=similarities.get)
    meilleur_user = next(u for u in tous_utilisateurs if u["name"] == meilleur_nom)

    deja_vus = {h["movie"] for h in utilisateur["watch_history"]}
    suggestions = [
        h for h in meilleur_user["watch_history"]
        if h["movie"] not in deja_vus and h["rating"] >= 4
    ]
    suggestions.sort(key=lambda h: -h["rating"])

    return {
        "utilisateur_similaire": meilleur_nom,
        "correlation": similarities[meilleur_nom],
        "suggestions": suggestions[:top_n]
    }


print("Filtrage collaboratif (Pearson)")
for u in utilisateurs:
    res = recommander_collaboratif(u, utilisateurs)
    print(f'{u["name"]} -> similaire a {res["utilisateur_similaire"]} (r = {res["correlation"]})')
    for s in res["suggestions"]:
        print(f'  - {s["movie"]:<25} rating : {s["rating"]}/5')
    print()


## 6. Clustering K-Means + Recommandation par groupe

`scipy.cluster.vq.kmeans` regroupe les utilisateurs selon leurs preferences de genre
(encodage binaire) et leur note moyenne.

**Amelioration cle** : apres le clustering, on recommande a chaque membre du groupe
les films les mieux notes par les autres membres du meme cluster (films non encore vus).


In [ ]:
def clustering_et_recommandation(utilisateurs, films, k=2, top_n=3):
    genres_uniques = sorted({g for u in utilisateurs for g in u["preferences"]})

    vecteurs = []
    for u in utilisateurs:
        vec = [1 if g in u["preferences"] else 0 for g in genres_uniques]
        vec.append(np.mean([h["rating"] for h in u["watch_history"]]))
        vecteurs.append(vec)

    data = np.array(vecteurs, dtype=float)
    blanchis = whiten(data)
    centroides, _ = kmeans(blanchis, k, seed=42)
    labels, _ = vq(blanchis, centroides)

    # Regrouper les utilisateurs par cluster
    clusters_membres = {}
    for i, u in enumerate(utilisateurs):
        clusters_membres.setdefault(int(labels[i]), []).append(u)

    # Pour chaque cluster, identifier les films les mieux notes du groupe
    resultats = {}
    for cluster_id, membres in clusters_membres.items():
        notes_groupe = {}
        for u in membres:
            for h in u["watch_history"]:
                notes_groupe.setdefault(h["movie"], []).append(h["rating"])

        top_films_groupe = sorted(
            [(t, round(np.mean(ns), 2)) for t, ns in notes_groupe.items()],
            key=lambda x: -x[1]
        )[:top_n]

        resultats[cluster_id] = {
            "membres": [u["name"] for u in membres],
            "top_films": top_films_groupe
        }

    return resultats, labels


resultats_clusters, labels_users = clustering_et_recommandation(utilisateurs, films, k=2)

print("Clustering K-Means (k=2) - Composition des groupes")
print("=" * 55)
for cluster_id, info in resultats_clusters.items():
    print(f"  Cluster {cluster_id + 1} : {', '.join(info['membres'])}")
    print(f"  Top films du groupe :")
    for titre, note_moy in info["top_films"]:
        print(f"    - {titre:<25} rating moyen groupe : {note_moy}/5")
    print()

print("Recommandations personnalisees par cluster :")
print("=" * 55)
for i, u in enumerate(utilisateurs):
    cluster_id = int(labels_users[i])
    info = resultats_clusters[cluster_id]
    deja_vus = {h["movie"] for h in u["watch_history"]}
    suggestions = [(t, n) for t, n in info["top_films"] if t not in deja_vus]
    print(f"{u['name']} (Cluster {cluster_id + 1}) :")
    if suggestions:
        for titre, note_moy in suggestions:
            print(f"  -> {titre:<25} (top du groupe, rating moy : {note_moy}/5)")
    else:
        print("  -> Tous les top films du groupe deja vus")
    print()


## 7. Analyses statistiques SciPy

- **Chi-carre** : la distribution des genres regardes est-elle uniforme ?
- **ANOVA** : les notes varient-elles significativement selon les genres ?
- **Regression lineaire** : quelle est la tendance des notes dans le temps ?


In [ ]:
# --- Chi-carre ---
genre_counts = {}
for u in utilisateurs:
    for h in u["watch_history"]:
        genre_counts[h["genre"]] = genre_counts.get(h["genre"], 0) + 1

observed = list(genre_counts.values())
expected = [sum(observed) / len(observed)] * len(observed)
chi2, p_chi2 = chisquare(observed, f_exp=expected)

print("Chi-carre (distribution des genres)")
print(f"  Chi2 = {chi2:.3f}  p-value = {p_chi2:.4f}")
print(f"  -> {'Distribution non uniforme' if p_chi2 < 0.05 else 'Distribution uniforme'}")
print()

# --- ANOVA ---
notes_par_genre = {}
for u in utilisateurs:
    for h in u["watch_history"]:
        notes_par_genre.setdefault(h["genre"], []).append(h["rating"])

groupes = [v for v in notes_par_genre.values() if len(v) >= 2]
F, p_anova = f_oneway(*groupes)

print("ANOVA one-way (ratings par genre)")
print(f"  F = {F:.3f}  p-value = {p_anova:.4f}")
print(f"  -> {'Difference significative entre genres' if p_anova < 0.05 else 'Pas de difference significative'}")
print()

# --- Regression lineaire ---
print("Regression lineaire (tendance des ratings dans le temps)")
for u in utilisateurs:
    historique = sorted(u["watch_history"], key=lambda h: h["date"])
    dates = [datetime.strptime(h["date"], "%Y-%m-%d") for h in historique]
    debut = min(dates)
    x = np.array([(d - debut).days for d in dates], dtype=float)
    y = np.array([h["rating"] for h in historique], dtype=float)
    slope, intercept, r, p, _ = linregress(x, y)
    tendance = "hausse" if slope > 0.001 else ("baisse" if slope < -0.001 else "stable")
    print(f"  {u['name']:<12} pente = {slope:+.4f}  R2 = {r**2:.3f}  -> {tendance}")


## 8. Visualisations

Cinq graphiques :
1. Repartition des genres regardes (camembert)
2. Evolution des notes dans le temps avec tendance lineaire
3. Notes moyennes par utilisateur
4. Distribution des genres dans la base de films
5. *(nouveau)* Taille des clusters K-Means


In [ ]:
fig = plt.figure(figsize=(16, 12))
fig.suptitle("Systeme de Recommandation de Films - Analyse", fontsize=15, fontweight="bold", y=1.01)

COULEURS = ["#4c72b0", "#dd8452", "#55a868", "#c44e52", "#8172b3",
            "#937860", "#da8bc3", "#8c8c8c", "#ccb974", "#64b5cd"]

# --- Graphique 1 : Camembert des genres (premier utilisateur) ---
ax1 = fig.add_subplot(2, 3, 1)
u0 = utilisateurs[0]
genre_counts_u0 = {}
for h in u0["watch_history"]:
    genre_counts_u0[h["genre"]] = genre_counts_u0.get(h["genre"], 0) + 1

ax1.pie(genre_counts_u0.values(), labels=genre_counts_u0.keys(),
        colors=COULEURS[:len(genre_counts_u0)], autopct="%1.0f%%", startangle=90)
ax1.set_title(f"Genres regardes par {u0['name']}")

# --- Graphique 2 : Evolution des ratings dans le temps ---
ax2 = fig.add_subplot(2, 3, 2)
historique_trie = sorted(u0["watch_history"], key=lambda h: h["date"])
dates_u0 = [datetime.strptime(h["date"], "%Y-%m-%d") for h in historique_trie]
ratings_u0 = [h["rating"] for h in historique_trie]

debut = min(dates_u0)
x_num = np.array([(d - debut).days for d in dates_u0], dtype=float)
slope, intercept, r, _, _ = linregress(x_num, ratings_u0)
tendance_y = intercept + slope * x_num

ax2.plot(dates_u0, ratings_u0, "o-", color="#4c72b0", linewidth=2, markersize=7, label="Ratings")
ax2.plot(dates_u0, tendance_y, "--", color="#dd8452", linewidth=1.5,
         label=f"Tendance (R2={r**2:.2f})")
ax2.set_title(f"Evolution des ratings - {u0['name']}")
ax2.set_ylabel("Rating (/5)")
ax2.set_ylim(0, 5.5)
ax2.legend()
ax2.grid(axis="y", linestyle="--", alpha=0.4)
ax2.tick_params(axis="x", rotation=30)

# --- Graphique 3 : Ratings moyens par utilisateur ---
ax3 = fig.add_subplot(2, 3, 3)
noms = [u["name"] for u in utilisateurs]
moyennes = [np.mean([h["rating"] for h in u["watch_history"]]) for u in utilisateurs]

barres = ax3.bar(noms, moyennes, color=COULEURS[:len(noms)])
ax3.set_title("Rating moyen par utilisateur")
ax3.set_ylabel("Rating moyen (/5)")
ax3.set_ylim(0, 5.5)
ax3.grid(axis="y", linestyle="--", alpha=0.4)
for barre, val in zip(barres, moyennes):
    ax3.text(barre.get_x() + barre.get_width() / 2, barre.get_height() + 0.05,
             f"{val:.2f}", ha="center", va="bottom", fontsize=9)

# --- Graphique 4 : Distribution des genres dans la base ---
ax4 = fig.add_subplot(2, 3, 4)
genres_base = {}
for f in films:
    genres_base[f["genre"]] = genres_base.get(f["genre"], 0) + 1

genres_tries = sorted(genres_base, key=genres_base.get, reverse=True)
counts_tries = [genres_base[g] for g in genres_tries]

barres2 = ax4.bar(genres_tries, counts_tries, color=COULEURS[:len(genres_tries)])
ax4.set_title("Distribution des genres (base de films)")
ax4.set_ylabel("Nombre de films")
ax4.grid(axis="y", linestyle="--", alpha=0.4)
ax4.tick_params(axis="x", rotation=30)
for barre, cnt in zip(barres2, counts_tries):
    ax4.text(barre.get_x() + barre.get_width() / 2, barre.get_height() + 0.05,
             str(cnt), ha="center", va="bottom", fontsize=9)

# --- Graphique 5 : Taille des clusters K-Means ---
ax5 = fig.add_subplot(2, 3, 5)
cluster_labels = [f"Cluster {cid+1}\n({', '.join(info['membres'])})"
                  for cid, info in resultats_clusters.items()]
cluster_sizes = [len(info["membres"]) for info in resultats_clusters.values()]

ax5.bar(cluster_labels, cluster_sizes, color=["#4c72b0", "#dd8452"])
ax5.set_title("Taille des clusters K-Means")
ax5.set_ylabel("Nombre d'utilisateurs")
ax5.set_ylim(0, max(cluster_sizes) + 1)
ax5.grid(axis="y", linestyle="--", alpha=0.4)
for barre, val in zip(ax5.patches, cluster_sizes):
    ax5.text(barre.get_x() + barre.get_width() / 2, barre.get_height() + 0.05,
             str(val), ha="center", va="bottom", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig("rapport_recommandation.png", dpi=120, bbox_inches="tight")
plt.show()
print("Graphiques sauvegardes dans rapport_recommandation.png")
